 PHASE 1: SCHEME INGESTION & DATA LAYER

In [0]:
%pip install datasets sentence-transformers faiss-cpu gradio
dbutils.library.restartPython()

  Using cached datasets-4.8.4-py3-none-any.whl.metadata (19 kB)
  Using cached sentence_transformers-5.4.1-py3-none-any.whl.metadata (17 kB)
  Using cached faiss_cpu-1.13.2-cp310-abi3-manylinux_2_27_aarch64.manylinux_2_28_aarch64.whl.metadata (7.6 kB)
  Using cached gradio-6.13.0-py3-none-any.whl.metadata (17 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached xxhash-3.6.0-cp312-cp312-manylinux2014_aarch64.manylinux_2_17_aarch64.manylinux_2_28_aarch64.whl.metadata (13 kB)
  Using cached multiprocess-0.70.19-py312-none-any.whl.metadata (7.5 kB)
  Using cached transformers-5.6.2-py3-none-any.whl.metadata (33 kB)
  Using cached torch-2.11.0-cp312-cp312-manylinux_2_28_aarch64.whl.metadata (29 kB)
  Using cached brotli-1.2.0-cp312-cp312-manylinux2014_aarch64.manylinux_2_17_aarch64.manylinux_2_28_aarch64.whl.metadata (6.1 kB)
  Using cached gradio_client-2.5.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached groovy-0.1.2-py3-none-any.whl.metadata (6.1 kB)
  Using 

In [0]:
%pip install -U fsspec huggingface_hub datasets
dbutils.library.restartPython()

  Attempting uninstall: fsspec
    Found existing installation: fsspec 2023.5.0
    Not uninstalling fsspec at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-5cc87444-aeda-4151-b54d-1fdc73b0c3c8
    Can't uninstall 'fsspec'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%pip install python-dotenv kaggle
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
from datasets import load_dataset
from pyspark.sql import functions as F
import os
from dotenv import load_dotenv

In [0]:
# Set Kaggle credentials temporarily in the environment
# os.environ['KAGGLE_USERNAME'] = dbutils.secrets.get(scope="hackathon", key="kaggle_user")
# os.environ['KAGGLE_KEY'] = dbutils.secrets.get(scope="hackathon", key="kaggle_user_key")


# Dynamically construct the path to the .env file in the current workspace directory
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
workspace_dir = "/Workspace" + "/".join(notebook_path.split("/")[:-1])
env_path = f"{workspace_dir}/.env"

# Load the variables
load_dotenv(env_path) # load_dotenv("/Workspace/Users/you/app/.env")

# Safety Check
if not os.getenv("KAGGLE_USERNAME") or not os.getenv("KAGGLE_KEY"):
    raise ValueError("Missing Kaggle credentials. Please check your .env file.")

print(f"Authenticated as: {os.getenv('KAGGLE_USERNAME')}")


Authenticated as: saipavanb


In [0]:
!mkdir -p /tmp/raw_data/
!kaggle datasets download -d jainamgada45/indian-government-schemes --unzip -p /tmp/raw_data/

Dataset URL: https://www.kaggle.com/datasets/jainamgada45/indian-government-schemes
License(s): CC0-1.0
100%|██████████████████████████████████████| 3.10M/3.10M [00:00<00:00, 48.3MB/s]



In [0]:
# Verify the download using shell command
!ls -lh /tmp/raw_data/

total 13M
-rw-r--r-- 1 spark-0896573f-36b5-4516-9382-7f spark-0896573f-36b5-4516-9382-7f 13M Apr 25 09:32 updated_data.csv


In [0]:
# Let's verify the exact filename extracted by Kaggle
print("Files in /tmp/raw_data/:", os.listdir("/tmp/raw_data/"))

Files in /tmp/raw_data/: ['updated_data.csv']


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS nyaya_hackathon;
CREATE SCHEMA IF NOT EXISTS nyaya_hackathon.schemes_app;
USE CATALOG nyaya_hackathon;
USE SCHEMA schemes_app;

In [0]:
# 1. Ingest gov_myscheme from Kaggle
print("Loading gov_myscheme dataset...")
local_csv_path = "/tmp/raw_data/updated_data.csv" 
pdf = pd.read_csv(local_csv_path)

cols_to_drop = [col for col in pdf.columns if 'unnamed' in str(col).lower()]
pdf = pdf.drop(columns=cols_to_drop)

# 3. Convert to Spark DataFrame
df_raw = spark.createDataFrame(pdf)

# 4. Standardize column names (removes spaces, makes lowercase)
df_clean = df_raw.toDF(*[c.replace(' ', '_').lower() for c in df_raw.columns])


# 5. Handle Nulls for Concatenation
# If we don't fill nulls, concat_ws will destroy the entire string if one column is missing
text_columns = [
    "scheme_name", "details", "benefits", 
    "eligibility", "schemecategory", "tags", "level"
]
df_filled = df_clean.fillna("", subset=text_columns)

Loading gov_myscheme dataset...


In [0]:
df_clean.columns

['scheme_name',
 'slug',
 'details',
 'benefits',
 'eligibility',
 'application',
 'documents',
 'level',
 'schemecategory',
 'tags']

In [0]:
df_clean.limit(10).display()

scheme_name slug details benefits eligibility application documents level schemecategory tags "Immediate Relief Assistance" under "Welfare and Relief for Fishermen During Lean Seasons and Natural Calamities Scheme" ira-wrflsncs The scheme "Immediate Relief Assistance" is a Sub-Component under the scheme "Welfare and Relief for Fishermen During Lean Seasons and Natural Calamities Scheme". The scheme is extended to all the regions of the Union territory of Puducherry. The scheme is introduced with the objective of extending financial assistance to the fishermen's families to compensate for the loss due to the missing breadwinner and to support them financially to run their family. ₹ 1,00,000, in two installments of ₹ 50,000 each, as immediate relief assistance for the family (legal heir) of the missing fisherman. ﻿ Disbursal Initially, 50% will be extended within 3 months from the date of receipt of the application from the family (legal heir). The family (legal heir) should approach this department for the release of the balance 50% of the relief which will be deposited in the bank in a joint account in the name of kin (legal heir) and the competent authority concerned. If no further information is received about the missing person, the balance amount will be released in favour of the next of kin (legal heir), after the prescribed period of 9 months from the date of release of 1st part of lump sum. *In case of the return of the missing fishermen, the amount extended as compensation either ₹ 50,000 or ₹ 1,00,000 as the case may be, will be recovered by invoking an insurance bond. The applicant should be the family (legal heir) of the missing fisherman. The missing fisherman should have been a resident of the Union territory of Puducherry. The missing fisherman must have lost his/her life while fishing. The missing fisherman must have been in the age group of 18-60 years. The missing fisherman must not have been a beneficiary of the old age pension scheme. The missing fisherman should have enrolled as a member of the Fishermen/Fisherwomen Co-operative Society. Step 1: The interested applicant should visit the office of the concerned authority i.e. the Department of Fisheries and Fishermen Welfare/Sub-Offices of outlying regions in all four regions. Step 1: The interested applicant should request the hard copy of the prescribed format of the application form from the concerned authority. Step 2: In the application form, fill in all the mandatory fields, paste the passport-sized photograph (signed across, if required), and attach copies of all the mandatory documents (self-attest, if required). Step 3: Submit the duly filled and signed application form along with the documents to the concerned authority. Step 4: Request a receipt or acknowledgment from the concerned authority to whom the application has been submitted. Ensure that the receipt contains essential details such as the date and time of submission, and a unique identification number (if applicable). *The affected family (legal heir) should apply immediately within 30 days from the date of the event for consideration. Photograph of the Family (Legal Heir) of the Missing Fisherman. Residential Certificate of the Missing Fishermen. Proof of Age of the Missing Fishermen. Declaration That the Missing Fisherman Was Not a Beneficiary of the Old Age Pension Scheme. Membership Certificate from the President/Administrator of Fishermen/Fisherwoman Co-operative Society. Electoral Identity Card (Attested Copy) Ration Card (Attested Copy) Panchayathar’s Letter of Village Concerned Name, Relationship, and Address of the Legal Heir (Affidavit in ₹ 5 Stamp Paper, Affixing as the Legal Heir Duly Signed in Before a Notary Public). ‘No Claim’ Certificate in Respect of Financial Assistance Extended by the Revenue Department Should Be Obtained From the Department of Revenue and Disaster Management. FIR and Non-traceable Certificate From Respective Station House Officer of Police Department. S

In [0]:
df_filled.limit(10).display()

scheme_name slug details benefits eligibility application documents level schemecategory tags "Immediate Relief Assistance" under "Welfare and Relief for Fishermen During Lean Seasons and Natural Calamities Scheme" ira-wrflsncs The scheme "Immediate Relief Assistance" is a Sub-Component under the scheme "Welfare and Relief for Fishermen During Lean Seasons and Natural Calamities Scheme". The scheme is extended to all the regions of the Union territory of Puducherry. The scheme is introduced with the objective of extending financial assistance to the fishermen's families to compensate for the loss due to the missing breadwinner and to support them financially to run their family. ₹ 1,00,000, in two installments of ₹ 50,000 each, as immediate relief assistance for the family (legal heir) of the missing fisherman. ﻿ Disbursal Initially, 50% will be extended within 3 months from the date of receipt of the application from the family (legal heir). The family (legal heir) should approach this department for the release of the balance 50% of the relief which will be deposited in the bank in a joint account in the name of kin (legal heir) and the competent authority concerned. If no further information is received about the missing person, the balance amount will be released in favour of the next of kin (legal heir), after the prescribed period of 9 months from the date of release of 1st part of lump sum. *In case of the return of the missing fishermen, the amount extended as compensation either ₹ 50,000 or ₹ 1,00,000 as the case may be, will be recovered by invoking an insurance bond. The applicant should be the family (legal heir) of the missing fisherman. The missing fisherman should have been a resident of the Union territory of Puducherry. The missing fisherman must have lost his/her life while fishing. The missing fisherman must have been in the age group of 18-60 years. The missing fisherman must not have been a beneficiary of the old age pension scheme. The missing fisherman should have enrolled as a member of the Fishermen/Fisherwomen Co-operative Society. Step 1: The interested applicant should visit the office of the concerned authority i.e. the Department of Fisheries and Fishermen Welfare/Sub-Offices of outlying regions in all four regions. Step 1: The interested applicant should request the hard copy of the prescribed format of the application form from the concerned authority. Step 2: In the application form, fill in all the mandatory fields, paste the passport-sized photograph (signed across, if required), and attach copies of all the mandatory documents (self-attest, if required). Step 3: Submit the duly filled and signed application form along with the documents to the concerned authority. Step 4: Request a receipt or acknowledgment from the concerned authority to whom the application has been submitted. Ensure that the receipt contains essential details such as the date and time of submission, and a unique identification number (if applicable). *The affected family (legal heir) should apply immediately within 30 days from the date of the event for consideration. Photograph of the Family (Legal Heir) of the Missing Fisherman. Residential Certificate of the Missing Fishermen. Proof of Age of the Missing Fishermen. Declaration That the Missing Fisherman Was Not a Beneficiary of the Old Age Pension Scheme. Membership Certificate from the President/Administrator of Fishermen/Fisherwoman Co-operative Society. Electoral Identity Card (Attested Copy) Ration Card (Attested Copy) Panchayathar’s Letter of Village Concerned Name, Relationship, and Address of the Legal Heir (Affidavit in ₹ 5 Stamp Paper, Affixing as the Legal Heir Duly Signed in Before a Notary Public). ‘No Claim’ Certificate in Respect of Financial Assistance Extended by the Revenue Department Should Be Obtained From the Department of Revenue and Disaster Management. FIR and Non-traceable Certificate From Respective Station House Officer of Police Department. S

In [0]:
# 6. Engineer the Vector Search Context
# We explicitly order this from high-level categorization down to specific details
df_silver = df_filled.withColumn(
    "search_context", 
    F.concat_ws(
        " | ", 
        F.col("scheme_name"),
        F.col("schemecategory"),
        F.col("level"),
        F.col("tags"),
        F.col("details"),
        F.col("eligibility")
    )
)

# 7. Write to Delta Table (Databricks Lakehouse architecture)
print("Writing structured data to Delta Lake...")
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true") # Forces schema update if you rerun
    .saveAsTable("nyaya_hackathon.schemes_app.silver_schemes")
)

print(" Success: Delta Table 'silver_schemes' is live and optimized for RAG.")

Writing structured data to Delta Lake...
 Success: Delta Table 'silver_schemes' is live and optimized for RAG.


In [0]:
# Verify the final schema and data
display(spark.table("nyaya_hackathon.schemes_app.silver_schemes").select("scheme_name", "search_context").limit(3))

scheme_name,search_context
Ladli Beti Scheme,"Ladli Beti Scheme | Banking,Financial Services and Insurance, Social welfare & Empowerment, Women and Child | State | Girl Child, Financial Assistance, Social Welfare | A social assistance scheme sponsored by the Jammu & Kashmir Government for newborn girl child of the Union Territories of J&K and Ladakh born on or after 01/04/2015. The objective of the scheme is to arrest the declining female sex ratio. The scheme further intends to ensure that the girl child does not become a burden for the parent or guardian at the time of her marriage. The scheme is a hybrid deposit plan having two phases: ﻿ Phase I: A recurring deposit for 14 years having a date of completion as one month after the last installment received in the account. Phase II: A Cumulative Term Deposit (CCR) for 7 years. ﻿ After the maturity of Phase I (recurring deposit account) the account will graduate to Phase II (Cumulative Term Deposit account). The monthly contribution of ₹1000/- in Phase-I is made by the J&K Govt. The annual income of the parents from all sources should be less than ₹75,000. The required documents are the Application Form, KYC Norms of the Parent/ Guardian, and the sanction letter from the CDPO (Child Development Project Officer). | The girl child should have been born on or after 01/04/2015. The annual income of the parents of the girl child from all sources should be less than ₹75,000. The girl child and her family should be residents of Jammu & Kashmir."
Ladli Laxmi Yojana,"Ladli Laxmi Yojana | Social welfare & Empowerment, Women and Child | State | Financial Assistance, Empowerment, Social Welfare, Girl Child | Ladli Lakshmi Yojana has been implemented in Madhya Pradesh with the objective of creating positive thinking in public towards the birth of girl child, improvement in sex ratio, improvement in educational level, health condition of girls and laying the foundation stone for their good future. ﻿ Some families are financially weak and not able to give higher education to their daughters nor can they collect money for their marriage. The State government has started the Ladli Laxmi Yojana to provide financial assistance for the education and marriage of the daughter. This money can be used by the girl for her higher education or marriage and to promote the empowerment of women in the state. | For Normal Cases The girl child should be born on or after January 1, 2006. The girl child should be registered in the local Anganwadi center. Parents should be natives of Madhya Pradesh. Parents should not be income taxpayers. Family planning has been adopted after the birth of the second child. Parents who have two or less than two children, family planning has been adopted after the birth of the second child. The girl child born from the first delivery will be given benefits without family planning but It is necessary for the mother/father to adopt family planning in order to get the benefit for the second girl child. ﻿ For Special Cases In a family where there are maximum 02 children and the mother or father has died, registration can be done till the girl child is 05 years old. But in this type of case, if the woman/man gets married for the second time and already has 02 children, then the daughter born from the second marriage will not get the benefit of the scheme. If there are 03 girls together born at the time of first delivery, all three girls will get the benefit. Girls born to women prisoners in jail will also be benefited under the scheme. The benefit of the scheme will be given to a girl child born to a rape victim or a girl child. The collector has been given the authority to approve the cases till 02 instead of 01 year in the cases where family planning has not been adopted due to health related reasons. District collector will provide approval/rejection under special cases, after minutely examining the applications received late. The application has to be made by the Superintendent of